# utterancer-e2e-standalone-dev

End-to-end standalone dev version with all code in one place for development work

## Check models loadable in current env

In [2]:
# Verify models loadable in current environment
from silero_vad import load_silero_vad, get_speech_timestamps, read_audio

vad_model = load_silero_vad()

In [7]:
import torch
from ctc_forced_aligner import load_alignment_model

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}  dtype: {dtype}")

alignment_model, alignment_tokenizer = load_alignment_model(device, dtype=dtype)

Device: cuda  dtype: torch.float16


Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

# Pre-processing: resample and loudnorm

- Just pre-process both files here and skip if already exists
- Pre-process with EBU R128 broadcast loudness target and to 16 kHz mono for Silero, Gemini, and MMS

In [37]:
%%bash
for path_stem in \
    "data/eng_obama-unga71/20160920_POTUS_Opening_of_71st_Session_General_Assembly" \
    "data/hin_ekka-premchand-do-sakhiyan-ch9/dosakhiyan_09_premchand_128kb"; do

    # ── FLAC: loudnorm + 16kHz mono. Source of truth for all downstream stages. ──
    if [[ -f "${path_stem}.flac" ]]; then
        echo "Skip: ${path_stem}.flac exists"
    else
        ffmpeg -loglevel error -hide_banner -nostats -nostdin \
            -i "${path_stem}.mp3" \
            -ac 1 -af loudnorm=I=-16:TP=-1.5:LRA=11 -ar 16000 \
            "${path_stem}.flac"
    fi

    # ── Opus: derived from FLAC. Inspector-only; ~10x smaller than the FLAC. ──
    # Encoded from the already-loudnormed FLAC so its loudness matches the
    # canonical pipeline audio. -application voip optimises for speech;
    # 64kbps is transparent for mono speech at this bandwidth.
    if [[ -f "${path_stem}.opus" ]]; then
        echo "Skip: ${path_stem}.opus exists"
    else
        ffmpeg -loglevel error -hide_banner -nostats -nostdin \
            -i "${path_stem}.flac" \
            -c:a libopus -b:a 64k -application voip -map_metadata -1 \
            "${path_stem}.opus"
    fi
done

Skip: data/eng_obama-unga71/20160920_POTUS_Opening_of_71st_Session_General_Assembly.flac exists
Skip: data/eng_obama-unga71/20160920_POTUS_Opening_of_71st_Session_General_Assembly.opus exists
Skip: data/hin_ekka-premchand-do-sakhiyan-ch9/dosakhiyan_09_premchand_128kb.flac exists
Skip: data/hin_ekka-premchand-do-sakhiyan-ch9/dosakhiyan_09_premchand_128kb.opus exists


# Set processing variables

In [43]:
from pathlib import Path

RECORDING_PATH = Path("data/eng_obama-unga71/20160920_POTUS_Opening_of_71st_Session_General_Assembly.flac")

## Silero VAD: detect time intervals with voice activity

In [46]:
from silero_vad import load_silero_vad, get_speech_timestamps, read_audio

model = load_silero_vad()

In [47]:
recording_samples   = read_audio(RECORDING_PATH, sampling_rate=16000)
recording_duration  = recording_samples.shape[0]/16_000

fragments_intervals = get_speech_timestamps(
    recording_samples,
    model,
    return_seconds=True,
    # Force Silero to cut to defend against rapid speech recordings
    max_speech_duration_s=60
)

In [48]:
fragments_intervals[:5]

[{'start': 0.2, 'end': 0.5},
 {'start': 0.9, 'end': 2.0},
 {'start': 3.1, 'end': 4.4},
 {'start': 5.7, 'end': 6.8},
 {'start': 7.9, 'end': 8.9}]

## Chapteriser: bisect a recording into chapters at the longest fragment gaps

### Algorithm

1. Treat the whole recording as one chapter.
2. Find the longest chapter. If it's within the size budget, we're done.
3. Otherwise, find the longest fragment gap *inside* that chapter and cut there.
4. Repeat.

This is priority-queue tree-splitting: the priority is "chapter is over-length"
and the split point is "the most acoustically defensible cut available." The
longest gap is, by definition, the location where cutting interferes least with
the recording's prosody — wherever the speaker paused longest.

### The zero-duration gap edge case

Setting `max_speech_duration_s=60` on Silero forces it to emit fragment
boundaries inside any continuous speech span longer than 60s, even when
there's no real silence to cut on. The resulting fragments are sometimes
back-to-back with essentially zero gap between them. Silero's
`use_max_poss_sil_at_max_speech=True` default softens this — it searches for
the quietest sub-100ms dip within the forced region and cuts there — but the
gap reported between fragments can still be at or near zero.

The chapteriser handles this by **keeping every fragment gap as a candidate
cut**, regardless of duration. When real pauses exist they win on duration;
when they don't, zero-duration gaps remain valid candidates and ties get
broken by proximity to the chapter centre. This produces balanced halves in
the uniform continuous-speech case while still preferring real pauses
wherever they exist.

In [50]:
from dataclasses import dataclass


@dataclass
class FragmentGap:
    """A candidate chapter boundary. Real pauses have duration > 0; forced
    cuts inside continuous speech may have duration ≈ 0 but are still valid."""
    start: float
    end: float

    @property
    def midpoint(self) -> float:
        return (self.start + self.end) / 2

    @property
    def duration(self) -> float:
        return self.end - self.start


def fragment_gaps_from_fragments(fragments_intervals, recording_duration):
    """Complement of the fragments: gaps before, between, and after."""
    gaps = []
    prev_end = 0.0
    for frag in fragments_intervals:
        # Use >= not > so zero-duration boundaries (forced cuts in continuous
        # speech) are kept as valid candidates.
        if frag["start"] >= prev_end:
            gaps.append(FragmentGap(prev_end, frag["start"]))
        prev_end = frag["end"]
    if recording_duration >= prev_end:
        gaps.append(FragmentGap(prev_end, recording_duration))
    return gaps


def chapterise(fragments_intervals, recording_duration,
               max_chapter_sec=600, min_chapter_sec=60):
    """Bisect at the longest fragment gaps until every chapter fits the budget.
    Then merge any chapter shorter than `min_chapter_sec` into the neighbour
    it's prosodically closer to (the side with the shorter bounding gap).

    The unified principle: pauses drive every decision. Long pauses become
    chapter boundaries; short pauses are where adjacent chapters belong
    together.
    """
    gaps = fragment_gaps_from_fragments(fragments_intervals, recording_duration)
    # Index gaps by midpoint so we can look up the gap that produced any
    # given chapter boundary.
    gap_by_midpoint = {g.midpoint: g for g in gaps}

    # ── Bisection: cut at the longest internal gap until under budget ──
    chapters = [(0.0, recording_duration)]
    while True:
        longest_idx = max(range(len(chapters)),
                          key=lambda i: chapters[i][1] - chapters[i][0])
        start, end = chapters[longest_idx]
        if end - start <= max_chapter_sec:
            break
        internal = [g for g in gaps if start < g.midpoint < end]
        if not internal:
            break    # nothing to cut on; accept the over-length chapter
        # Prefer the longest gap; tie-break by proximity to chapter centre.
        chapter_mid = (start + end) / 2
        best = max(internal,
                   key=lambda g: (g.duration, -abs(g.midpoint - chapter_mid)))
        chapters[longest_idx:longest_idx + 1] = [
            (start, best.midpoint), (best.midpoint, end),
        ]

    chapters = sorted(chapters)

    # ── Merge step: absorb any chapter shorter than min_chapter_sec into the
    # neighbour it's prosodically more connected to (shorter bounding gap). ──
    changed = True
    while changed:
        changed = False
        for i, (s, e) in enumerate(chapters):
            if e - s >= min_chapter_sec:
                continue

            # First chapter has no left neighbour; last has no right.
            # Otherwise compare the gaps that bound this chapter and merge
            # toward the shorter one (where the speaker was most connected).
            if i == 0:
                merge_direction = "right"
            elif i == len(chapters) - 1:
                merge_direction = "left"
            else:
                left_gap_dur  = gap_by_midpoint[s].duration
                right_gap_dur = gap_by_midpoint[e].duration
                merge_direction = "left" if left_gap_dur <= right_gap_dur else "right"

            if merge_direction == "left":
                chapters = (chapters[:i - 1]
                            + [(chapters[i - 1][0], e)]
                            + chapters[i + 1:])
            else:
                chapters = (chapters[:i]
                            + [(s, chapters[i + 1][1])]
                            + chapters[i + 2:])
            changed = True
            break    # restart enumeration after mutation

    return chapters

In [51]:
chapter_intervals = chapterise(fragments_intervals, recording_duration, max_chapter_sec=600)

print(f"Recording: {recording_duration:.1f}s ({recording_duration/60:.1f} min)")
print(f"Fragments: {len(fragments_intervals)}")
print(f"Chapters:  {len(chapter_intervals)}")
print()
for i, (s, e) in enumerate(chapter_intervals):
    print(f"  chapter {i:>2}: {s:>7.1f} -> {e:>7.1f}s  ({e-s:>5.1f}s)")

Recording: 2868.4s (47.8 min)
Fragments: 795
Chapters:  7

  chapter  0:     0.0 ->   215.3s  (215.3s)
  chapter  1:   215.3 ->   621.1s  (405.8s)
  chapter  2:   621.1 ->  1115.8s  (494.7s)
  chapter  3:  1115.8 ->  1443.2s  (327.4s)
  chapter  4:  1443.2 ->  1986.5s  (543.2s)
  chapter  5:  1986.5 ->  2490.9s  (504.4s)
  chapter  6:  2490.9 ->  2868.4s  (377.5s)


### Write chapters out to disk

In [66]:
import soundfile as sf

# Save each chapter
for i, (start, end) in enumerate(chapter_intervals):
    sample_start = int(start * 16_000)
    sample_end = int(end * 16_000)
    chapter_path = RECORDING_PATH.parent / (RECORDING_PATH.stem + f"___ch{i:02d}-{start:.2f}s--{end:.2f}s.flac")
    sf.write(chapter_path, recording_samples[sample_start:sample_end], 16_000)

## Gemini ASR: transcribe each chapter into a punctuated silver transcript

We send each chapter audio file to Gemini with a strict prompt that asks for:

- Verbatim transcription in the recording's script (Latin for English, Devanagari for Hindi).
- Sentence boundaries marked with **only** `.`, `?`, `!` (English) or `।`, `?`, `!` (Hindi).
- No commas, semicolons, em-dashes, parentheses, or other internal punctuation.
- Numbers spelled out as words.

This punctuation discipline is the load-bearing decision of the whole pipeline. Gemini doubles as both ASR and **sentence segmentation** here — it decides where utterances end based on the audio's prosody, grammar, and semantics simultaneously. The sentence-end marks survive through MMS forced alignment (the aligner ignores punctuation during phoneme matching but preserves it in word-level output), so the eventual utterance boundaries fall wherever Gemini placed a `.` / `?` / `!` / `।`. No threshold tuning, no per-genre calibration, no LLM follow-up call.

### Language switching

Set `LANGUAGE = "eng"` or `"hin"` at the top of the cell. The corresponding prompt is selected from the `PROMPTS` dict. Adding a new language is a single dict entry — translate the prompt, ensure the sentence-ender list covers that language's punctuation conventions, and the downstream MMS+utterance steps work unchanged.

### Caching and idempotency

Each chapter's transcript is cached as `{chapter_stem}.txt` next to its `.flac`. Re-running the cell skips already-transcribed chapters, so iterating on downstream steps (prompt tweaks, retries, alignment fixes) doesn't incur repeated Gemini cost. Pass `force=True` to `transcribe_chapter` to refresh a specific chapter, or delete the `.txt` files to retranscribe from scratch.

### Throttling

The free tier of Gemini 2.5 Flash allows 10 requests per minute. We enforce a 6-second floor between calls (`MIN_INTERVAL_SEC`) so a 7-chapter Obama run completes in ~50s without quota errors. If a 429 does come back, the retry logic reads the server-suggested wait time and sleeps accordingly.

### Output

Two artefact types per recording:

- `{stem}___chNN-AAA.AAs--BBB.BBs.txt` — per-chapter transcript, one file per chapter audio
- `{stem}___silver.txt` — full stitched transcript, one chapter per line

The silver file is what gets fed to MMS forced alignment in the next step. Per-chapter files exist for debugging (compare a chapter's audio to its transcript directly in the directory listing) and for re-running just one chapter when needed.

In [7]:
import os
import re
import time
from pathlib import Path

from google import genai
from google.genai.errors import ClientError


# ── Configuration ──────────────────────────────────────────────────────────
LANGUAGE = "eng"        # "eng" or "hin"
GEMINI_MODEL = "gemini-2.5-flash"
MIN_INTERVAL_SEC = 6.0  # ≥ 6s between calls keeps us under Flash's 10 RPM ceiling

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# ── Language-specific prompts ──────────────────────────────────────────────
PROMPTS = {
    "eng": """\
Transcribe this audio verbatim. Output rules:

- Mixed case (sentence-initial capitalisation, proper nouns, etc.).
- Use periods to end declarative sentences.
- Use question marks for questions.
- Use exclamation marks for emphatic sentences.
- DO NOT use commas, semicolons, em-dashes, parentheses, or any other punctuation.
  Only the three sentence-end marks above are permitted.
- Numbers spelled out as words: "twenty seven" not "27".
- If unsure of a word, omit it. Do not guess.

Output only the transcript. No preamble, no commentary.
""",
    "hin": """\
इस ऑडियो का शब्दशः लिप्यांतरण करें। नियम:

- सामान्य देवनागरी लिपि में लिखें।
- घोषणात्मक वाक्यों के अंत में पूर्ण विराम (।) का प्रयोग करें।
- प्रश्नवाचक वाक्यों के अंत में अंग्रेज़ी प्रश्न चिह्न (?) का प्रयोग करें।
- विस्मयादिबोधक वाक्यों के अंत में अंग्रेज़ी विस्मयादिबोधक चिह्न (!) का प्रयोग करें।
- अल्पविराम (,), अर्धविराम (;), डैश (—), कोष्ठक, या कोई अन्य विराम चिह्न का प्रयोग न करें।
  केवल ऊपर दिए गए तीन वाक्यांत चिह्न ही मान्य हैं।
- संख्याओं को शब्दों में लिखें: "सत्ताईस" लिखें, "27" नहीं।
- यदि किसी शब्द के बारे में सुनिश्चित न हों, तो उसे छोड़ दें। अनुमान न लगाएं।

केवल लिप्यांतरण ही दें। कोई प्रस्तावना या टिप्पणी नहीं।
""",
}

prompt = PROMPTS[LANGUAGE]


# ── Throttle + retry ────────────────────────────────────────────────────────
_last_call_at = 0.0

def _throttle():
    """Block until at least MIN_INTERVAL_SEC has passed since the last call."""
    global _last_call_at
    wait = MIN_INTERVAL_SEC - (time.time() - _last_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_call_at = time.time()


def call_with_retry(fn, max_attempts=5):
    """Retry on 429 using the server-supplied delay; exponential backoff otherwise."""
    for attempt in range(max_attempts):
        _throttle()
        try:
            return fn()
        except ClientError as e:
            if e.code != 429:
                raise
            m = re.search(r"retry in (\d+\.?\d*)s", str(e))
            wait = (float(m.group(1)) * 1.2) if m else (2 ** attempt) * 5.0
            print(f"  ⏳ 429; sleeping {wait:.1f}s  (attempt {attempt + 1}/{max_attempts})")
            time.sleep(wait)
    raise RuntimeError(f"Gave up after {max_attempts} attempts")


# ── Transcription per chapter ───────────────────────────────────────────────
def transcribe_chapter(chapter_path: Path, *, force: bool = False) -> str:
    """Transcribe a chapter audio file. Caches output as `{stem}.txt`
    alongside the input; re-run with force=True to retranscribe."""
    out_path = chapter_path.with_suffix(".txt")
    if out_path.exists() and not force:
        return out_path.read_text(encoding="utf-8")

    uploaded = client.files.upload(file=chapter_path)
    while uploaded.state.name == "PROCESSING":
        time.sleep(1)
        uploaded = client.files.get(name=uploaded.name)

    response = call_with_retry(lambda: client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[prompt, uploaded],
    ))
    transcript = response.text.strip()
    out_path.write_text(transcript, encoding="utf-8")
    print(f"  ✓ {chapter_path.name}  ({len(transcript.split())} words)")
    return transcript


In [10]:
# ── Discover and transcribe all chapter files for the current recording ────
chapter_files = sorted(
    RECORDING_PATH.parent.glob(f"{RECORDING_PATH.stem}___ch*.flac")
)
print(f"Transcribing {len(chapter_files)} chapter(s) for {RECORDING_PATH.stem}")

transcripts = {}
for chapter_path in chapter_files:
    transcripts[chapter_path.stem] = transcribe_chapter(chapter_path)

# ── Stitch the silver: one chapter per line, in chapter order ───────────────
silver_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___silver.txt")
silver_path.write_text(
    "\n".join(transcripts[p.stem] for p in chapter_files),
    encoding="utf-8",
)

n_sentence_enders = sum(
    transcripts[p.stem].count(c)
    for p in chapter_files
    for c in (".", "?", "!", "।")
)
total_words = sum(len(transcripts[p.stem].split()) for p in chapter_files)
print()
print(f"silver  → {silver_path.name}")
print(f"         {total_words} words across {n_sentence_enders} sentence-ending marks")

Transcribing 7 chapter(s) for 20160920_POTUS_Opening_of_71st_Session_General_Assembly
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch00-0.00s--215.30s.flac  (353 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch01-215.30s--621.10s.flac  (775 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch02-621.10s--1115.85s.flac  (1049 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch03-1115.85s--1443.25s.flac  (654 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch04-1443.25s--1986.50s.flac  (1084 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch05-1986.50s--2490.90s.flac  (1011 words)
  ✓ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___ch06-2490.90s--2868.41s.flac  (700 words)

silver  → 20160920_POTUS_Opening_of_71st_Session_General_Assembly___silver.txt
         5626 words across 352 sentence-ending marks


## MMS forced alignment: word-level timestamps on the silver

We now have a punctuated silver transcript and the original recording. MMS forced alignment produces a `(start, end, text, score)` record for every word in the silver. The aligner runs on the **full recording** rather than per-chapter — chapter splitting was a Gemini-context-window concern, not an MMS one, and the aligner handles 30+ minute inputs comfortably on GPU.

### What MMS does internally

The aligner romanises the silver via `uroman` (Devanagari → Latin transliteration for Hindi; pass-through for English) and matches each phoneme to acoustic frames using CTC alignment. Sentence-ending punctuation (`.`, `?`, `!`, `।`) is silently ignored during the phoneme-matching step but **preserved in the output word text** — so each MMS word still carries any trailing punctuation the silver had. That's how punctuation flows from Gemini through MMS into the utterance-emission step that comes next.

### Output

A single JSON file per recording, alongside the chapter and silver artefacts:

`{stem}___words.json` — list of `{start, end, text, score}` records, ordered by start time. This is the canonical word-level alignment from which utterance boundaries get extracted in the next step.

In [24]:
import json
import math
from pathlib import Path

import torch
from ctc_forced_aligner import (
    generate_emissions,
    get_alignments,
    get_spans,
    load_alignment_model,
    load_audio,
    postprocess_results,
    preprocess_text,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}  dtype: {dtype}")

alignment_model, alignment_tokenizer = load_alignment_model(device, dtype=dtype)

# Map our two-letter LANGUAGE codes to the ISO-639-3 codes uroman/MMS expect.
# Extend this dict to add new languages.
ISO_LANG = {"eng": "eng", "hin": "hin"}


# ── Load audio + silver ─────────────────────────────────────────────────────
silver_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___silver.txt")
silver = silver_path.read_text(encoding="utf-8").strip()

audio_waveform = load_audio(
    str(RECORDING_PATH),
    alignment_model.dtype,
    alignment_model.device,
)

# ── Generate phoneme emissions over the full recording ────────────────────
emissions, stride = generate_emissions(
    alignment_model,
    audio_waveform,
    # bump higher on bigger GPUs; 4 fits comfortably on T4
    batch_size=4,
)

# ── Tokenise the silver into phoneme tokens MMS can match against ─────────
tokens_starred, text_starred = preprocess_text(
    silver,
    romanize=True,
    language=ISO_LANG[LANGUAGE],
    star_frequency="segment",
)

# ── CTC alignment + word-span extraction ──────────────────────────────────
segments, scores, blank_token = get_alignments(
    emissions, tokens_starred, alignment_tokenizer,
)
spans = get_spans(tokens_starred, segments, blank_token)

# ── Reconstruct word-level records with original text (including punctuation) ──
word_timestamps = postprocess_results(text_starred, spans, stride, scores)


# ── Score normalisation: MMS emits log-probabilities; viewer expects 0-1 ──
def normalise_score(log_prob):
    """MMS log-prob (negative, ~-2 high-confidence to ~-15 low) → 0–1 sigmoid."""
    if log_prob is None:
        return None
    return 1.0 / (1.0 + math.exp(-(log_prob + 5) / 2))


words = [
    {
        "start": w["start"],
        "end":   w["end"],
        "text":  w["text"],
        "score": normalise_score(w.get("score")),
    }
    for w in word_timestamps
]

# ── Save ────────────────────────────────────────────────────────────────────
words_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___words.json")
words_path.write_text(
    json.dumps(words, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# ── Summary ────────────────────────────────────────────────────────────────
print(f"Aligned {len(words)} words across {audio_waveform.shape[-1] / 16_000:.1f}s of audio")
print()
print(f"First 3 words:")
for w in words[:3]:
    print(f"  {w['start']:6.2f}–{w['end']:6.2f}s  score={w['score']:.2f}  {w['text']!r}")
print(f"\nLast 3 words:")
for w in words[-3:]:
    print(f"  {w['start']:6.2f}–{w['end']:6.2f}s  score={w['score']:.2f}  {w['text']!r}")

avg_score = sum(w["score"] for w in words if w["score"] is not None) / len(words)
print(f"\nMean confidence: {avg_score:.2f}")
print(f"→ {words_path.name}")

Device: cuda  dtype: torch.float16


Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

Aligned 5626 words across 2868.4s of audio

First 3 words:
    0.88–  1.08s  score=0.74  'Mr.'
    1.14–  1.60s  score=0.32  'President.'
    3.12–  3.30s  score=0.70  'Mr.'

Last 3 words:
  2855.88–2855.92s  score=0.27  'you'
  2855.96–2856.06s  score=0.91  'very'
  2856.14–2868.40s  score=0.87  'much.'

Mean confidence: 0.85
→ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___words.json


## Utterance emission: punctuation proposes, acoustics dispose

We now have MMS word records with start/end timestamps and Gemini-emitted
punctuation preserved in the word text. The job is to walk this stream and
emit utterance boundaries.

Naively this could be "split at every `[.!?।]`" but two failure modes make
that wrong:

1. **Abbreviation false positives.** Gemini sometimes writes "Mr.", "U.N.",
   etc. despite prompt instructions to spell them out. A naive split treats
   "Mr." as a sentence end and emits a 200ms one-word utterance.

2. **Long compound sentences.** Some real sentences run 15-30 seconds with
   no internal punctuation. TTS training prefers utterances under ~15s. A
   naive emitter would produce these unbroken.

The principled solution uses three signals together:

| signal | role |
|---|---|
| **Sentence-end punctuation in word text** | primary candidate for a break |
| **Gap to next word** | distinguishes real periods from abbreviation periods |
| **Accumulated utterance duration** | force-split if no real punctuation arrives in time |

### Decision logic per candidate break

A `.` `?` `!` `।` at the end of a word triggers a *candidate* break. The
candidate becomes a *real* break unless the gap to the next word is below
`MIN_GAP_AFTER_PUNCT_SEC` (~200ms) — short gaps indicate the punctuation
was an abbreviation, not a sentence end.

Independently, if the accumulating utterance exceeds `MAX_UTTERANCE_SEC`
without a candidate break firing, we force-split at the longest internal
inter-word gap. This catches long compound sentences without disturbing
the linguistic-judgment-driven majority case.

A final pass merges any utterance shorter than `MIN_UTTERANCE_SEC` into
the closer-pause neighbour, eliminating fragments that survived the
above logic (very short emphatic statements not bounded by real pauses).

### Parameters

| parameter | default | reason |
|---|---|---|
| `MIN_GAP_AFTER_PUNCT_SEC` | 0.2 | Below this, period is treated as abbreviation. Above this, as sentence end. Empirically matches the bimodal pause distribution between mid-word transitions (<100ms) and inter-sentence pauses (200ms+). |
| `MAX_UTTERANCE_SEC` | 15.0 | TTS training upper bound for clean clips. Compound sentences above this get force-split. |
| `MIN_UTTERANCE_SEC` | 0.5 | Below this, the utterance is treated as a fragment and merged with a neighbour. Conservative — short emphatic utterances like "Yes." stay alive at 0.5s. |

In [52]:
import json
import re
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
# Aligned to prod values where applicable; new features (prosodically-closer
# merge direction, abbreviation filter, force-split) layered on top.
SENTENCE_END_PATTERNS = {
    "eng": re.compile(r"[.!?]$"),
    "hin": re.compile(r"[।?!]$"),
}

MIN_GAP_AFTER_PUNCT_SEC = 0.20    # below → abbreviation, suppress break (new feature)
MAX_UTTERANCE_SEC       = 15.0    # above → force-split (matches prod max_duration)
GAP_MERGE_THRESHOLD_SEC = 0.75    # below → merge neighbours (matches prod silence_threshold)

sentence_end_re = SENTENCE_END_PATTERNS[LANGUAGE]


# ── Load aligned word stream ───────────────────────────────────────────────
words_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___words.json")
words = json.loads(words_path.read_text(encoding="utf-8"))


# ── Helpers ────────────────────────────────────────────────────────────────
def gap_after(i):
    """Inter-word gap from word i's end to word i+1's start. None at end of list."""
    if i >= len(words) - 1:
        return None
    return words[i + 1]["start"] - words[i]["end"]


def make_utterance(span):
    """Build an utterance record from a list of word dicts."""
    text = " ".join(w["text"] for w in span)
    scored = [w["score"] for w in span if w.get("score") is not None]
    return {
        "start":     span[0]["start"],
        "end":       span[-1]["end"],
        "text":      text,
        "sentence":  text,        # viewer compatibility
        "words":     span,
        "avg_score": (sum(scored) / len(scored)) if scored else None,
    }


# ── Phase 1: split on confirmed punctuation breaks ─────────────────────────
# A confirmed break = sentence-end punctuation AND a sufficient gap to the
# next word. Spurious abbreviation periods get filtered here.
utterances = []
current = []

for i, w in enumerate(words):
    current.append(w)
    if not sentence_end_re.search(w["text"]):
        continue
    g = gap_after(i)
    if g is None:                          # last word: always end utterance
        break
    if g >= MIN_GAP_AFTER_PUNCT_SEC:       # real sentence end
        utterances.append(make_utterance(current))
        current = []
    # else: abbreviation; keep accumulating

if current:
    utterances.append(make_utterance(current))

print(f"Phase 1 (punctuation+gap): {len(utterances)} utterances")


# ── Phase 2: force-split overlong utterances at the longest internal gap ──
def split_long_utterance(utt, max_sec):
    """Recursively split at longest internal gap until all pieces are under max."""
    if utt["end"] - utt["start"] <= max_sec:
        return [utt]
    span = utt["words"]
    if len(span) < 2:
        return [utt]   # single word; can't split further
    gaps = [(i, span[i + 1]["start"] - span[i]["end"]) for i in range(len(span) - 1)]
    cut_idx, _ = max(gaps, key=lambda x: x[1])
    left  = make_utterance(span[:cut_idx + 1])
    right = make_utterance(span[cut_idx + 1:])
    return split_long_utterance(left, max_sec) + split_long_utterance(right, max_sec)


phase2 = []
for u in utterances:
    phase2.extend(split_long_utterance(u, MAX_UTTERANCE_SEC))
n_force_splits = len(phase2) - len(utterances)
utterances = phase2
print(f"Phase 2 (length cap):       {len(utterances)} utterances ({n_force_splits} forced splits)")


# ── Phase 3: merge across short inter-utterance gaps ──────────────────────
# Matches prod's silence_threshold rule but picks the merge direction by
# prosodic proximity (shorter gap wins) rather than always merging left.
# Skipped if merging would exceed MAX_UTTERANCE_SEC.
def merge_short_gaps(utts, gap_threshold, max_sec):
    """For each utterance, examine the gaps to its neighbours. If the
    shorter gap is below gap_threshold AND merging in that direction stays
    under max_sec, merge. Repeat to fixed point."""
    result = list(utts)
    changed = True
    while changed:
        changed = False
        for i, u in enumerate(result):
            left_gap  = u["start"] - result[i - 1]["end"] if i > 0 else float("inf")
            right_gap = result[i + 1]["start"] - u["end"] if i < len(result) - 1 else float("inf")

            # Pick the closer neighbour
            if left_gap <= right_gap:
                neighbour_gap, direction = left_gap, "left"
            else:
                neighbour_gap, direction = right_gap, "right"

            if neighbour_gap >= gap_threshold:
                continue    # gap is large enough that they're separate utterances

            # Check the merged duration wouldn't exceed the cap
            if direction == "left":
                merged_dur = u["end"] - result[i - 1]["start"]
            else:
                merged_dur = result[i + 1]["end"] - u["start"]
            if merged_dur > max_sec:
                continue    # would over-merge; leave alone

            if direction == "left":
                merged = make_utterance(result[i - 1]["words"] + u["words"])
                result = result[:i - 1] + [merged] + result[i + 1:]
            else:
                merged = make_utterance(u["words"] + result[i + 1]["words"])
                result = result[:i] + [merged] + result[i + 2:]
            changed = True
            break
    return result


utterances = merge_short_gaps(utterances, GAP_MERGE_THRESHOLD_SEC, MAX_UTTERANCE_SEC)
print(f"Phase 3 (gap merge):        {len(utterances)} utterances")

# ── Phase 4: pairwise boundary expansion ──────────────────────────────────
# MMS edges are tight; pad both sides by a fixed offset, sharing the gap
# midpoint when wanted-bounds would otherwise overlap.
START_OFFSET = -0.150   # seconds; negative → push start earlier
END_OFFSET   = +0.150   # seconds; positive → push end later

def expand_utterance_bounds(utts, recording_duration,
                            start_offset=START_OFFSET,
                            end_offset=END_OFFSET):
    """Compute new (start, end) for every utterance using original bounds
    as input, then apply them all at once. This avoids order-dependence:
    each pair's decision uses the original (pipeline-produced) gap, not
    a half-mutated state."""
    n = len(utts)
    if n == 0:
        return utts

    new_starts = [None] * n
    new_ends   = [None] * n

    # First utterance start: clamp to 0
    new_starts[0] = max(0.0, utts[0]["start"] + start_offset)

    # Pairwise: for each adjacent (i, i+1), set utts[i].end and utts[i+1].start
    for i in range(n - 1):
        want_end_curr   = utts[i]["end"]     + end_offset
        want_start_next = utts[i + 1]["start"] + start_offset

        if want_end_curr <= want_start_next:
            # No conflict; both get their full offset.
            new_ends[i]       = want_end_curr
            new_starts[i + 1] = want_start_next
        else:
            # Encroachment; share the original gap at its midpoint.
            mid = (utts[i]["end"] + utts[i + 1]["start"]) / 2
            new_ends[i]       = mid
            new_starts[i + 1] = mid

    # Last utterance end: clamp to recording_duration
    new_ends[n - 1] = min(recording_duration, utts[n - 1]["end"] + end_offset)

    # Safety check: every utterance should still have positive duration.
    # If not (very short utt with offsets resolved to a single point on
    # both sides), fall back to the original bounds for that utterance.
    n_clamped = 0
    for i in range(n):
        if new_starts[i] >= new_ends[i]:
            new_starts[i] = utts[i]["start"]
            new_ends[i]   = utts[i]["end"]
            n_clamped += 1
    if n_clamped:
        print(f"  Phase 4: {n_clamped} utterance(s) reverted to original "
              f"bounds (offsets would have produced zero/negative duration)")

    # Apply
    for i in range(n):
        utts[i]["start"] = new_starts[i]
        utts[i]["end"]   = new_ends[i]
    return utts

utterances = expand_utterance_bounds(utterances, recording_duration)
print(f"Phase 4 (boundary expand):  {len(utterances)} utterances "
      f"(start_offset={START_OFFSET*1000:+.0f}ms, end_offset={END_OFFSET*1000:+.0f}ms)")

# ── Save ───────────────────────────────────────────────────────────────────
out_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___utterances.json")
out_path.write_text(
    json.dumps(utterances, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# ── Summary ────────────────────────────────────────────────────────────────
durations = sorted(u["end"] - u["start"] for u in utterances)
n = len(durations)
mean_dur = sum(durations) / n

print()
print(f"Final: {n} utterances over {durations[-1]:.1f}s longest, {durations[0]:.2f}s shortest")
print(f"       median {durations[n // 2]:.2f}s, mean {mean_dur:.2f}s")

buckets = [(0, 2), (2, 5), (5, 10), (10, 15), (15, 999)]
print(f"\nDistribution:")
for lo, hi in buckets:
    c = sum(1 for d in durations if lo <= d < hi)
    bar = "█" * int(100 * c / n / 2)
    label = f"{lo:>2}–{hi if hi < 999 else '∞':>3}s"
    print(f"  {label}  {c:>4} ({100 * c / n:>5.1f}%)  {bar}")

print(f"\n→ {out_path.name}")

Phase 1 (punctuation+gap): 342 utterances
Phase 2 (length cap):       359 utterances (17 forced splits)
Phase 3 (gap merge):        345 utterances
Phase 4 (boundary expand):  345 utterances (start_offset=-150ms, end_offset=+150ms)

Final: 345 utterances over 15.3s longest, 0.38s shortest
       median 5.18s, mean 6.12s

Distribution:
   0–  2s    44 ( 12.8%)  ██████
   2–  5s   120 ( 34.8%)  █████████████████
   5– 10s   127 ( 36.8%)  ██████████████████
  10– 15s    49 ( 14.2%)  ███████
  15–  ∞s     5 (  1.4%)  

→ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___utterances.json


## Romanisation: add Latin-script forms to each utterance

For non-Latin-script languages (Hindi, Arabic, Russian, etc.), downstream
consumers often need a romanised view of the utterance text — for
search-ability, for users who can't read the original script, or for
audio-text alignment debugging in tools that expect Latin characters.

We use `uroman`, the same romaniser MMS uses internally for forced
alignment. This guarantees consistency: the Latin form an utterance
gets here is the same form MMS used when aligning the phonemes. No
surprises from competing transliteration standards.

### Behaviour by language

| `LANGUAGE` | uroman lcode | what happens |
|---|---|---|
| `"eng"` | `"eng"` | pass-through; Latin script stays Latin |
| `"hin"` | `"hin"` | Devanagari → Latin per ISO 15919 (with vowelisation) |
| Other | from `ISO_LANG` dict | as configured |

For English the romanisation field will be identical to the original
text, which is fine — having the field present uniformly across
languages simplifies downstream code that doesn't want to special-case
script-by-script.

### What we're updating

This step **reads** `___utterances.json`, adds a `text_romanised` field
to each record, and **writes back to the same file**. Idempotent — running
twice produces the same result. Already-romanised utterances are
detected via field presence and skipped, so re-running after
incrementally adding utterances is cheap.

In [53]:
import json
from pathlib import Path

import uroman as ur


# ── Load utterances ─────────────────────────────────────────────────────────
utterances_path = RECORDING_PATH.with_name(RECORDING_PATH.stem + "___utterances.json")
utterances = json.loads(utterances_path.read_text(encoding="utf-8"))


# ── Romanise ────────────────────────────────────────────────────────────────
# Constructor loads uroman data tables (~1s). Single instance reused for all
# utterances. lcode is ISO-639-3; we already have ISO_LANG mapping from the
# MMS step.
romaniser = ur.Uroman()
lcode = ISO_LANG[LANGUAGE]

n_updated = 0
for u in utterances:
    if "text_romanised" in u:
        continue       # idempotent: already done
    u["text_romanised"] = romaniser.romanize_string(u["text"], lcode=lcode)
    n_updated += 1


# ── Write back ──────────────────────────────────────────────────────────────
utterances_path.write_text(
    json.dumps(utterances, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# ── Summary ─────────────────────────────────────────────────────────────────
print(f"Romanised {n_updated} of {len(utterances)} utterances "
      f"(lcode={lcode!r}, skipped {len(utterances) - n_updated} already-romanised)")
print()
print("Examples:")
for u in utterances[:3]:
    print(f"  {u['text'][:70]!r}")
    print(f"    → {u['text_romanised'][:70]!r}")
print(f"\n→ {utterances_path.name}")

Romanised 345 of 345 utterances (lcode='eng', skipped 0 already-romanised)

Examples:
  'Mr. President.'
    → 'Mr. President.'
  'Mr. Secretary General.'
    → 'Mr. Secretary General.'
  'Fellow delegates.'
    → 'Fellow delegates.'

→ 20160920_POTUS_Opening_of_71st_Session_General_Assembly___utterances.json
